# Stiffness — a.k.a. "the thing that makes my simulation slow"

Stiffness is one of the main reasons why numerical simulations are slow.
It is also surprisingly hard to define: there is no single accepted mathematical definition, but several related mechanisms that make a PDAE system behave this way.

**TODO (hook plot):** two simulations of the same model with the same number of unknowns, one 50× slower than the other, differing only in the binding rate constants. *Which one is the big model?* — neither.

## What is it

A system is stiff when the time integrator is forced into **much smaller time steps than the result you look at would require**, because something in the system is much faster than what you actually care about.
Smaller steps mean more computations and therefore longer simulations.

Explicit methods are forced into small steps by **stability**: above a critical step size the solution blows up.
Implicit methods have no such limit — they are forced into small steps only by **accuracy** and by the nonlinear solve. They pay for that with extra work per step: each step is a Newton solve.
Whenever the explicit stability limit is severe, paying more per step beats taking many more steps, which is why implicit integration is the standard choice for chromatography. CADET uses it throughout, so stiffness does not break your simulation — it bills you for it.

**TODO:** plot the stability regions of two explicit methods, plot the eigenvalues of a chromatography system and show how the step size scales them into (or out of) the stability region. Then: implicit BDF is stable everywhere in the left half plane, so only accuracy dictates its step size.

<details>
<summary><b>Click to learn more: stability vs. accuracy, explicit vs. implicit</b></summary>

**Instability** means the accumulation of approximation errors until the simulation "explodes" — values grow or drop without bound.
**Inaccuracy** means an under-resolved solution: stable, plausible in order of magnitude, and simply wrong.
Both give you a useless result, but they fail differently, and the cure differs too: refining an implicit solution improves it monotonically, while an explicit solution stays garbage until the step size crosses the stability limit and only then starts to improve.

**TODO:** minimal example with two plots — left, a blown-up explicit solution; right, a stable but completely inaccurate implicit solution. Then the same two under refinement.

</details>

<details>
<summary><b>Click to learn more: the textbook example</b></summary>

The classic illustration is a linear system with eigenvalues $-1$ and $-1000$.
The fast component has decayed to nothing after a few thousandths of a second, and the answer you want is governed by the slow one. An explicit method still has to respect $h < 2/1000$ for the rest of the simulation, purely for stability — it keeps resolving something that is no longer there.

**TODO:** three lines of numpy, two curves.

</details>

## Where does it come from

Mathematically, the explicit step-size restriction — our working indicator of stiffness — is tied to the **eigenvalues** of the discretized system, which say how fast the different parts of the solution evolve.
A large eigenvalue means a fast process, and the integrator has to resolve it: for stability if it is explicit, for accuracy and nonlinear convergence if it is implicit. Either way the whole simulation slows down.

In chromatography, there are seven recurring sources:

| # | Cause | Mechanism | Who controls it |
|---|-------|-----------|-----------------|
| 1 | **Different time scales** | Adsorption or chemical reactions can be almost instantaneous, while pore diffusion and convection take much longer. The integrator keeps resolving the fast process long after it has settled into quasi-equilibrium. | model |
| 2 | **Different length scales** | Particle radius ≈ 45 µm against a bed length of ≈ 1.4 cm. Diffusion times scale with $L^2/D$, so the particle equilibrates orders of magnitude faster than the column. Things simply happen faster in small geometries. | model |
| 3 | **Steep gradients** | Self-sharpening Langmuir fronts, step injections, breakthrough. Here it is not the equations but the *current solution* that causes the trouble. | model + feed |
| 4 | **The grid itself** | The discretization is what turns spatial derivatives into eigenvalues, and they grow as you refine: roughly $u/\Delta x$ for convection, $D/\Delta x^2$ for dispersion, $p^2/\Delta x$ for DG of degree $p$. Refining therefore makes the system stiffer, and the cost grows faster than the number of unknowns. | **user** |
| 5 | **Flow and geometry** | The interstitial velocity sets the convective time scale, and it is higher at smaller cross sections. Radial-flow and frustum columns have a velocity that varies along the bed, so the tightest limit is set locally. | model |
| 6 | **Events** | Every section transition and valve switch is a discontinuity. The integrator restarts at low order with a tiny step size and has to climb back up, plus a consistent initialization each time. Gradient staircases and SMB switching pay that price over and over. | user |
| 7 | **Scaling** | Salt at $10^2$–$10^3$ mol m$^{-3}$ next to proteins at $10^{-3}$. A single global `ABSTOL` either over-resolves the protein or ignores it, and a badly scaled Jacobian makes the Newton solve struggle. | user |

Two things we should be honest about:

- **#3 and #7 are not stiffness in the strict mathematical sense.** They are accuracy and nonlinear-convergence problems. But the symptoms — small time steps, many Newton iterations, long runtimes — are identical, and so is your experience of them.
- **Dispersion cuts both ways.** It raises the diffusive eigenvalues, which is stiffer, but it also smooths out steep fronts, which helps with #3. Which effect wins is a question for the experiment, not for the lecture.

### What is *not* stiffness, but also makes it slow

Worth naming, so that nobody misdiagnoses: the sheer size of the model, the number of components and particle types, the number of output time points and writing out the full bulk/particle solution, parameter sensitivities, running single-threaded, and an algorithmically differentiated instead of an analytic Jacobian.

## How do I see it?

A feeling that "it is slow" is not a diagnosis. What we can measure:

1. **Wall-clock time** — the blunt instrument, and the one every user already has.
2. **Number of internal time steps, and the step size over simulated time.** This is the telling picture: mark the section boundaries and you see both the restarts and the integrator crawling through a steep front.
3. **Newton convergence failures and error-test failures** from the CADET log — these separate "stiff" from "nonlinearly hard".
4. **Rule of thumb:** internal steps per second of simulated time. If that is far above what the shape of your chromatogram requires, something fast is being resolved that you do not care about.

**TODO:** check what CADET actually returns here. With `USER_SOLUTION_TIMES` set, the returned times are the ones you requested, not the integrator's own steps. Then collect it in `utility/stiffness_metrics.py` so that every notebook of this chapter measures the same way.

## What can I do about it

| Cause | First thing to try | What it costs you |
|-------|--------------------|-------------------|
| fast binding kinetics (#1) | rapid equilibrium, `IS_KINETIC = 0` | you get a DAE instead of an ODE, so consistent initialization matters; only valid if the kinetics really are fast |
| particle resolution (#2) | LRMP or LRM instead of GRM | model fidelity — there is no pore gradient any more |
| steep fronts (#3) | FV with WENO, or DG with oscillation suppression; dispersion if you can justify it physically | accuracy, or model fidelity if you add dispersion you cannot justify |
| the grid (#4) | find the coarsest grid that still meets your accuracy target — **do this before anything else** | nothing. This is free money, and most users leave it on the table |
| tolerances | match `ABSTOL`/`RELTOL` to the accuracy you actually need, and to the magnitudes of your components | accuracy, honestly accounted for |
| many sections (#6) | merge sections, or use a smooth polynomial gradient instead of a staircase | a slightly different feed profile |
| scaling (#7) | component-appropriate tolerances and sane units | nothing |

<details>
<summary><b>Click to learn more: specialized time integration, and why it is not in CADET</b></summary>

Specialized methods can exploit the separation of time scales. **IMEX** methods treat the stiff processes implicitly and the rest explicitly; **multirate** methods advance different processes with different step sizes.
Both need additional knowledge of the model structure and careful handling of the coupling between processes, which is hard to provide in a general-purpose code — although CADET's modular structure would be a reasonable starting point for future work in that direction.

IMEX methods have been demonstrated for chromatography models in the literature. They have not established a general performance advantage over conventional implicit integrators such as adaptive BDF, which remains the standard choice in established chromatography simulators, and no open-source implementation is available.

**TODO:** proper citations.
- https://www.sciencedirect.com/science/article/pii/S0168927417301812
- https://www.sciencedirect.com/science/article/pii/S0098135417304362
- https://www.sciencedirect.com/science/chapter/bookseries/abs/pii/B9780444642417501373

</details>

### Key takeaway

> **A slow simulation is not necessarily a big simulation.** A small model with one very fast process, one very steep front, or one needlessly fine grid can cost more than a large, well-behaved one.

## What comes next

One notebook per source of stiffness. Each sets up a chromatography problem that exercises exactly that mechanism, follows the same path — **setting → the one knob → measure → mitigate → honest limitation** — and ends with what you cannot fix.

| Notebook | Setting | The knob |
|----------|---------|----------|
| `Stiffness 1 – Time scales` | LWE / SMA, GRM | binding rate constants (equilibrium unchanged), then `IS_KINETIC` |
| `Stiffness 2 – Steep fronts` | 2-component Langmuir, LRM | `COL_DISPERSION`, sharpness of the feed profile |
| `Stiffness 3 – Length scales` | LWE, GRM vs. LRMP vs. LRM | `PAR_RADIUS`, `PORE_DIFFUSION`, transport model |
| `Stiffness 4 – The grid` | LWE, DG and FV | number of cells, polynomial degree |
| `Stiffness 5 – Events and scaling` | LWE with its three sections | number and length of sections, `ABSTOL` vs. component magnitudes |

Not every simulation can be made fast. Where that is the case, we say so.